# Case 3: Newton continuation with the section $x_1=y_1$

The Newton fixed-point calculation is kept in the same form as the original notebook. The only structural change before the failure analysis is the Poincaré section:

\[
g(q)=x_1-y_1=0,
\]

and we take the return with

\[
\dot g=p_{x_1}-p_{y_1}>0.
\]

The extra code after Newton is only for diagnosing **Newton non-convergence**. A state with no real momentum on the requested energy surface is treated as an invalid energy choice, not as the Newton failure studied here.


In [30]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root
from functools import partial

from system import System, inverse_power_potential, kinetic
from energy_continuation import (
    Section,
    _pack_section_state,
    _unpack_section_state,
    _poincare_map,
    _poincare_jacobian,
    _newton_fixed_point,
    _first_return,
    _rescale_momentum_guess,
    _total_energy,
)


## system and Case 3 initial orbit


In [31]:
PARAMETERS = {
    "k": 1.0,
    "alpha": 1.0,
    "beta": 1.0,
    "C": 1.0,
    "Lx": 10.0,
    "Ly": 10.0,
}
MASSES = np.array([1.0, 1.0])

# Original report01 Case 3 state: use it only as the energy reference.
a = 3.6644771408428354
A0 = 0.001
Q_REFERENCE = np.array([
    [a + A0 / 2, a + A0 / 2],
    [10.0 - a - A0 / 2, 10.0 - a - A0 / 2],
])
P_REFERENCE = np.array([
    [0.1, -0.1],
    [-0.1, 0.1],
])

# Match the final report04 example: a nearby symmetric Newton seed at the
# same total energy as the original Case 3 state.
REFERENCE_ENERGY = _total_energy(
    Q_REFERENCE, P_REFERENCE, kinetic, inverse_power_potential, PARAMETERS, MASSES
)
b = 3.6
Q_STAR = np.array([[b, b], [10.0 - b, 10.0 - b]])
seed_speed = np.sqrt(
    (REFERENCE_ENERGY - inverse_power_potential(Q_STAR, **PARAMETERS)) / 2.0
)
P_STAR = np.array([
    [seed_speed, -seed_speed],
    [-seed_speed, seed_speed],
])


## section coordinates

The Poincare section uses `energy_continuation`'s `Section`/pack/unpack machinery.
The return crossing is located by bisecting the leapfrog step
(`energy_continuation._first_return`). The numerical controls below match the
final Case 3 example in report04: `h=0.0025` and `max_return_steps=100000`.

Section: $g=x_1-y_1=0$ with $\dot g=p_{x_1}-p_{y_1}>0$, encoded as
`Section(particle=0, coordinate=1, normal=(1, -1), value=0, direction=1)`.
The six independent coordinates are $(u_1, x_2, y_2, p_{x_1}, p_{x_2}, p_{y_2})$
with $u_1=(x_1+y_1)/\sqrt2$ the coordinate along the section plane; $H=E$ still
removes one momentum coordinate (the component along the section normal).


In [32]:
# z = (u1, x2, y2, px1, px2, py2); u1=(x1+y1)/sqrt(2), y1=x1 is imposed by the section.
SECTION_COORDINATES = ("u1", "x2", "y2", "px1", "px2", "py2")
MAX_RETURN_STEPS = 100_000

CASE3_SECTION = Section(particle=0, coordinate=1, normal=(1.0, -1.0), value=0.0, direction=1)

system = System(
    q0=Q_STAR.copy(), p0=P_STAR.copy(), KE=kinetic,
    V=inverse_power_potential, V_parameters=PARAMETERS, masses=MASSES,
)
FORCE = system.force


def total_energy(q, p):
    """H=T+V: total Hamiltonian energy (energy_continuation._total_energy)."""
    return _total_energy(q, p, kinetic, inverse_power_potential, PARAMETERS, MASSES)


def pack_section_state(q, p):
    """Keep the six independent coordinates on the Case 3 section."""
    return _pack_section_state(q, p, CASE3_SECTION)


def unpack_section_state(z, energy):
    """Recover (q,p); H=E determines the omitted (section-normal) momentum."""
    return _unpack_section_state(
        z, energy, Q_STAR.shape, CASE3_SECTION, MASSES, inverse_power_potential, PARAMETERS
    )


## Poincare return map

Integrate to the next crossing of $g=x_1-y_1=0$ with $\dot g=p_{x_1}-p_{y_1}>0$,
using `energy_continuation._first_return` (bisection-refined crossing, see above).


In [33]:
def first_positive_return(force, masses, q0, p0, h, max_steps=MAX_RETURN_STEPS):
    """Return to the Case 3 section (energy_continuation._first_return)."""
    return _first_return(force, masses, q0, p0, h, CASE3_SECTION, max_steps)


def poincare_map(z, energy, force, masses, h, max_return_steps=MAX_RETURN_STEPS):
    """P_E(z): one section point mapped to its next return (energy_continuation._poincare_map)."""
    return _poincare_map(
        z, energy, force, masses, Q_STAR.shape, CASE3_SECTION,
        inverse_power_potential, PARAMETERS, h, max_return_steps,
    )


## numerical Jacobian $DP_E(z)$

Central finite-difference Jacobian, via `energy_continuation._poincare_jacobian`.


In [34]:
def poincare_jacobian(z_star, energy, force, masses, h, relative_step=1e-6):
    """Approximate DP_E(z_star) by central differences (energy_continuation._poincare_jacobian)."""
    pm = partial(poincare_map, energy=energy, force=force, masses=masses, h=h)
    return _poincare_jacobian(pm, z_star, relative_step)


## Newton method

Newton solves

\[
F(z)=P_E(z)-z=0, \qquad (DP-I)\,\Delta z=-F,
\]

via `energy_continuation._newton_fixed_point`. It adds a backtracking line
search on top of the plain Newton step (damping the correction until the
residual actually decreases), which keeps trial states energetically
feasible; it still raises `RuntimeError` on non-convergence or a singular
`DP-I`, and the continuation routine below catches that to diagnose the
first failure instead of stopping with an uncaught error.


In [35]:
def newton_fixed_point(
    z_guess, energy, force, masses, h,
    tol=1e-9, max_iter=10, relative_step=1e-6,
):
    """Newton iteration for F(z)=P_E(z)-z=0 (energy_continuation._newton_fixed_point)."""
    pm = partial(poincare_map, energy=energy, force=force, masses=masses, h=h)
    z_star, _, _ = _newton_fixed_point(pm, z_guess, tol, max_iter, relative_step, verbose=True)
    return z_star


## initial guess at the next energy

Same momentum-rescaling predictor as report04, via
`energy_continuation._rescale_momentum_guess`. The target energies are exactly
`E0 * [1.000, 0.999, ..., 0.995]`, matching report04's final Case 3 example.


In [36]:
def rescale_momentum_guess(z, old_energy, new_energy):
    """Rescale momentum only to predict the next energy (energy_continuation._rescale_momentum_guess)."""
    return _rescale_momentum_guess(
        z, old_energy, new_energy, Q_STAR.shape, CASE3_SECTION,
        MASSES, kinetic, inverse_power_potential, PARAMETERS,
    )


## quantities stored before a Newton failure

For every successfully converged orbit we store:

\[
d_\lambda=\min_i|\lambda_i-1|,
\qquad
\kappa=\operatorname{cond}(DP-I),
\qquad
T(E)=\text{return time}.
\]

A fold/multiplier candidate should show $d_\lambda\downarrow0$ and $\kappa\uparrow$. A homoclinic candidate should show $T(E)\uparrow$; near a saddle equilibrium we additionally test whether the orbit point approaches that saddle.


In [37]:
def orbit_diagnostics(z_star, energy, h, relative_step):
    """Compute the three small diagnostics needed later if Newton fails."""
    DP = poincare_jacobian(z_star, energy, FORCE, MASSES, h, relative_step)
    # lambda_i=eig(DP): Floquet multipliers of the Poincare fixed point.
    multipliers = np.linalg.eigvals(DP)
    # min|lambda_i-1|: distance of the closest multiplier to +1.
    lambda_gap = np.min(np.abs(multipliers - 1.0))
    # cond(DP-I): ill-conditioning of the Newton matrix.
    condition = np.linalg.cond(DP - np.eye(len(z_star)))

    q_star, p_star = unpack_section_state(z_star, energy)
    # T(E): one-return time; for a fixed point this is the orbit period.
    period, _, _ = first_positive_return(FORCE, MASSES, q_star, p_star, h)

    return q_star, p_star, multipliers, lambda_gap, condition, period


def monotone(values, direction):
    """Simple trend check on the last few successful continuation points."""
    values = np.asarray(values, dtype=float)
    if len(values) < 3:
        return False
    differences = np.diff(values)
    return np.all(differences < 0) if direction == "down" else np.all(differences > 0)


def nearby_saddle(q_guess):
    """Find a nearby equilibrium F(q)=0 and keep it only if Hessian(V) has a negative eigenvalue."""
    # root(F)=0: an equilibrium has zero force, equivalently grad(V)=0.
    solution = root(lambda q: FORCE(q.reshape(2, 2)).ravel(), q_guess.ravel())
    if not solution.success:
        return None

    q_saddle = solution.x.reshape(2, 2)
    eps = 1e-5
    JF = np.empty((4, 4))
    for j in range(4):
        dq = np.zeros(4)
        dq[j] = eps
        # JF[:,j] ~= [F(q+eps e_j)-F(q-eps e_j)]/(2 eps).
        JF[:, j] = (
            FORCE((solution.x + dq).reshape(2, 2)).ravel()
            - FORCE((solution.x - dq).reshape(2, 2)).ravel()
        ) / (2 * eps)

    # Hessian(V)=-JF because F=-grad(V); a negative eigenvalue means an unstable/saddle direction.
    hessian_eigs = np.linalg.eigvalsh(-JF)
    return q_saddle if np.min(hessian_eigs) < 0.0 else None


## continuation and first-failure diagnosis

Diagnosis order follows the requested logic:

1. Retry with the **energy continuation step halved**. No Newton damping is used. If the midpoint and then the original target converge, the failure is classified as an energy-step-size failure.
2. Otherwise inspect the multiplier/fold trend.
3. Otherwise inspect the period and saddle-distance trend for a homoclinic candidate.
4. If none is supported, print `reason unclear`.


In [38]:
def try_newton_from_previous(z_previous, previous_energy, target_energy, h, relative_step):
    """Predict from the last successful orbit and run the unchanged Newton corrector."""
    guess = rescale_momentum_guess(z_previous, previous_energy, target_energy)
    return newton_fixed_point(guess, target_energy, FORCE, MASSES, h, relative_step=relative_step)


def plot_failure_trends(results, saddle=None):
    """Plot only the trends used to classify the first Newton failure."""
    if len(results) < 2:
        return

    E = np.array([r["energy"] for r in results])
    gap = np.array([r["lambda_gap"] for r in results])
    cond = np.array([r["condition"] for r in results])
    period = np.array([r["period"] for r in results])

    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ax[0].plot(E, gap, "o-")
    ax[0].set_title(r"$\min_i|\lambda_i-1|$")
    ax[0].set_xlabel("E")
    ax[1].semilogy(E, cond, "o-")
    ax[1].set_title(r"$\mathrm{cond}(DP-I)$")
    ax[1].set_xlabel("E")
    ax[2].plot(E, period, "o-")
    ax[2].set_title("T(E)")
    ax[2].set_xlabel("E")
    plt.tight_layout()
    plt.show()

    if saddle is not None:
        # distance=min particle-to-saddle-particle Euclidean distance in full configuration space.
        distance = np.array([np.linalg.norm(r["q_star"] - saddle) for r in results])
        plt.figure(figsize=(5, 4))
        plt.plot(E, distance, "o-")
        plt.xlabel("E")
        plt.ylabel("||q*-q_saddle||")
        plt.title("Distance to nearby saddle")
        plt.tight_layout()
        plt.show()


def diagnose_failure(results, failed_energy, z_previous, previous_energy, h, relative_step, failure_error):
    """Diagnose the first Newton non-convergence without turning unrelated energy errors into Newton failures."""
    print(f"\nFirst Newton failure at E = {failed_energy:.10f}")
    print(f"Original Newton exception: {type(failure_error).__name__}: {failure_error}")

    # E_mid=(E_prev+E_fail)/2: halve ONLY the continuation energy step.
    midpoint_energy = 0.5 * (previous_energy + failed_energy)
    print(f"Retry 1: halve the ENERGY continuation step, E_mid = {midpoint_energy:.10f}")

    try:
        z_mid = try_newton_from_previous(z_previous, previous_energy, midpoint_energy, h, relative_step)
        # Retry the original failed target from the successful midpoint.
        z_target = try_newton_from_previous(z_mid, midpoint_energy, failed_energy, h, relative_step)
        print("FAILURE REASON: continuation energy step was too large; half-step continuation succeeds.")
        return "energy step too large", z_target
    except ValueError as exc:
        print(f"Half-step energy-surface reconstruction failed: {type(exc).__name__}: {exc}")
    except RuntimeError as exc:
        print(f"Half-step test did not recover the target: {type(exc).__name__}: {exc}")
        print("Checking dynamical indicators.")

    recent = results[-5:]
    gaps = [r["lambda_gap"] for r in recent]
    conditions = [r["condition"] for r in recent]
    periods = [r["period"] for r in recent]

    # Fold candidate: closest multiplier approaches +1 while DP-I becomes more ill-conditioned.
    if monotone(gaps, "down") and monotone(conditions, "up"):
        print("FAILURE REASON: multiplier/fold candidate: min|lambda-1| decreases while cond(DP-I) increases.")
        plot_failure_trends(results)
        return "multiplier/fold candidate", None

    # Find a nearby equilibrium from the last successful periodic point and classify it by Hessian eigenvalues.
    saddle = nearby_saddle(results[-1]["q_star"]) if results else None
    distances = [np.linalg.norm(r["q_star"] - saddle) for r in recent] if saddle is not None else []

    # Homoclinic candidate: period grows and the periodic point approaches a nearby saddle equilibrium.
    if monotone(periods, "up") and saddle is not None and monotone(distances, "down"):
        print("FAILURE REASON: homoclinic candidate: T(E) increases and the periodic point approaches a nearby saddle.")
        plot_failure_trends(results, saddle)
        return "homoclinic candidate", None

    print("FAILURE REASON: unclear from the requested tests.")
    plot_failure_trends(results, saddle)
    return "reason unclear", None


def continue_periodic_orbit(energies, h=0.0025, relative_step=1e-6):
    """Continue Case 3 until the first Newton failure, then diagnose that failure."""
    z_star = pack_section_state(Q_STAR, P_STAR)
    previous_energy = energies[0]
    results = []

    for index, energy in enumerate(energies):
        print(f"\nE = {energy:.10f}")

        try:
            if index > 0:
                z_star = rescale_momentum_guess(z_star, previous_energy, energy)

            z_star = newton_fixed_point(
                z_star, energy, FORCE, MASSES, h, relative_step=relative_step,
            )
        except ValueError as exc:
            # Explicitly exclude the non-real-momentum issue from this study.
            print(f"STOP: invalid energy choice ({exc}). This is not a Newton non-convergence diagnosis.")
            break
        except RuntimeError as exc:
            if not results:
                print(f"FAILURE REASON: {type(exc).__name__}: {exc}")
                print("Newton failed at the initial energy, so there is no continuation history for dynamical trend tests.")
                break
            diagnose_failure(
                results, energy, results[-1]["z_star"], previous_energy,
                h, relative_step, exc,
            )
            break

        q_star, p_star, multipliers, gap, condition, period = orbit_diagnostics(
            z_star, energy, h, relative_step,
        )
        # residual=||P(z*)-z*||: final fixed-point accuracy.
        residual = np.linalg.norm(poincare_map(z_star, energy, FORCE, MASSES, h) - z_star)

        print(f"    residual = {residual:.3e}")
        print(f"    min|lambda-1| = {gap:.3e}")
        print(f"    cond(DP-I) = {condition:.3e}")
        print(f"    T(E) = {period:.8f}")

        results.append({
            "energy": energy,
            "q_star": q_star.copy(),
            "p_star": p_star.copy(),
            "z_star": z_star.copy(),
            "multipliers": multipliers.copy(),
            "lambda_gap": gap,
            "condition": condition,
            "period": period,
            "residual": residual,
        })
        previous_energy = energy

    return results


## Main

The scan uses the same multiplicative energy ratios, initial seed, integration step, and return-step limit as the final Case 3 example in report04. It stops at the first Newton failure and prints the underlying exception before running the continuation diagnostics.

In [39]:
def main():
    E0 = total_energy(Q_STAR, P_STAR)

    # Match the final report04 Case 3 continuation exactly.
    energy_ratios = np.array([1.000, 0.999, 0.998, 0.997, 0.996, 0.995])
    energies = E0 * energy_ratios

    print("section coordinates:", SECTION_COORDINATES)
    print("initial seed q:\n", Q_STAR)
    print("initial seed p:\n", P_STAR)
    print(f"initial Case 3 energy: {E0:.10f}")
    print("energy ratios:", energy_ratios)
    return continue_periodic_orbit(energies, h=0.0025, relative_step=1e-6)


if __name__ == "__main__":
    results = main()


section coordinates: ('u1', 'x2', 'y2', 'px1', 'px2', 'py2')
initial seed q:
 [[3.6 3.6]
 [6.4 6.4]]
initial seed p:
 [[ 0.09747491 -0.09747491]
 [-0.09747491  0.09747491]]
initial Case 3 energy: 2.0076519650
energy ratios: [1.    0.999 0.998 0.997 0.996 0.995]

E = 2.0076519650
    Newton  0: ||P(z)-z|| = 2.744e-02
    Newton  1: ||P(z)-z|| = 6.962e-04
    Newton  2: ||P(z)-z|| = 3.991e-07
    Newton  3: ||P(z)-z|| = 1.762e-13
    residual = 1.762e-13
    min|lambda-1| = 1.562e+00
    cond(DP-I) = 1.044e+01
    T(E) = 101.82023789

E = 2.0056443130
    Newton  0: ||P(z)-z|| = 4.163e-02
    Newton  1: ||P(z)-z|| = 8.830e-03
    Newton  2: ||P(z)-z|| = 7.572e-05
    Newton  3: ||P(z)-z|| = 2.185e-08
    Newton  4: ||P(z)-z|| = 7.004e-14
    residual = 7.004e-14
    min|lambda-1| = 8.347e-01
    cond(DP-I) = 6.729e+01
    T(E) = 109.22169744

E = 2.0036366611
    Newton  0: ||P(z)-z|| = 2.274e-01
    Newton  1: ||P(z)-z|| = 5.252e-02
    Newton  2: ||P(z)-z|| = 8.701e-03
    Newton  3: |